# Lab 2: Building an Analytical Base Table for Machine Learning

**Prepared by:** Sharad Laad
**LinkedIn:** https://www.linkedin.com/in/sharadlaad/

## 15. Task 2: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 16. Task 3: Define Folder Paths

In [ ]:
DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
FIGURES_DIR = Path("../figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for file in DATA_DIR.glob("*.csv"):
    print(file.name)

## 17. Task 4: Load Required Tables

In [ ]:
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
category_translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

In [ ]:
tables = {
    "orders": orders,
    "customers": customers,
    "reviews": reviews,
    "payments": payments,
    "order_items": order_items,
    "products": products,
    "category_translation": category_translation
}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

## 18. Task 5: Inspect Key Tables

In [ ]:
orders.head()

In [ ]:
customers.head()

In [ ]:
payments.head()

In [ ]:
order_items.head()

In [ ]:
reviews.head()

In [ ]:
products.head()

## 19. Task 6: Understand the Level of Each Table

| Table | Row Level |
| --- | --- |
| customers | One row per customer-order relationship |
| orders | One row per order |
| payments | One or more rows per order |
| order_items | One or more rows per order |
| reviews | Usually one row per order, but duplicates may exist |
| products | One row per product |
| category_translation | One row per product category |

## 20. Task 7: Convert Date Columns

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [ ]:
orders[date_columns].info()

## 21. Task 8: Create Date-Based Features

In [ ]:
orders["order_year"] = orders["order_purchase_timestamp"].dt.year
orders["order_month"] = orders["order_purchase_timestamp"].dt.month
orders["order_day"] = orders["order_purchase_timestamp"].dt.day
orders["order_day_of_week"] = orders["order_purchase_timestamp"].dt.dayofweek
orders["order_hour"] = orders["order_purchase_timestamp"].dt.hour

In [ ]:
orders[[
    "order_purchase_timestamp",
    "order_year",
    "order_month",
    "order_day",
    "order_day_of_week",
    "order_hour"
]].head()

## 22. Task 9: Create Delivery-Based Features

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - 
    orders["order_purchase_timestamp"]
).dt.days

orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"] - 
    orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - 
    orders["order_estimated_delivery_date"]
).dt.days

In [ ]:
orders["is_late_delivery"] = np.where(
    orders["delivery_delay_days"] > 0,
    1,
    0
)

In [ ]:
orders[[
    "order_id",
    "order_status",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days",
    "is_late_delivery"
]].head()

## 23. Task 10: Analyze Late Delivery Distribution

In [ ]:
orders["is_late_delivery"].value_counts()

In [ ]:
orders["is_late_delivery"].value_counts(normalize=True) * 100

In [ ]:
orders["is_late_delivery"].value_counts().plot(kind="bar")
plt.title("Late Delivery Distribution")
plt.xlabel("Is Late Delivery")
plt.ylabel("Number of Orders")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lab02_late_delivery_distribution.png")
plt.show()

## 24. Task 11: Aggregate Payments Data

In [ ]:
payment_features = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    max_payment_installments=("payment_installments", "max"),
    payment_types_count=("payment_type", "nunique")
).reset_index()

In [ ]:
payment_features.head()

## 25. Task 12: Add Dominant Payment Type

In [ ]:
dominant_payment_type = payments.groupby("order_id")["payment_type"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).reset_index()

dominant_payment_type = dominant_payment_type.rename(
    columns={"payment_type": "dominant_payment_type"}
)

In [ ]:
payment_features = payment_features.merge(
    dominant_payment_type,
    on="order_id",
    how="left"
)

## 26. Task 13: Aggregate Order Items Data

In [ ]:
item_features = order_items.groupby("order_id").agg(
    total_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    unique_products=("product_id", "nunique"),
    unique_sellers=("seller_id", "nunique")
).reset_index()

In [ ]:
item_features.head()

## 27. Task 14: Add Product Category Information

In [ ]:
items_products = order_items.merge(
    products,
    on="product_id",
    how="left"
)

In [ ]:
items_products = items_products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [ ]:
items_products[[
    "order_id",
    "product_id",
    "product_category_name",
    "product_category_name_english"
]].head()

## 28. Task 15: Create Main Product Category Per Order

In [ ]:
order_category = items_products.groupby("order_id").agg(
    main_product_category=(
        "product_category_name_english",
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    )
).reset_index()

In [ ]:
order_category.head()

## 29. Task 16: Prepare Review Features

In [ ]:
review_features = reviews.groupby("order_id").agg(
    review_score=("review_score", "mean"),
    review_comment_count=("review_comment_message", lambda x: x.notnull().sum()),
    has_review_comment=("review_comment_message", lambda x: int(x.notnull().any()))
).reset_index()

In [ ]:
review_features.head()

## 30. Task 17: Create the Analytical Base Table

In [ ]:
abt = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

abt = abt.merge(
    review_features,
    on="order_id",
    how="left"
)

abt = abt.merge(
    payment_features,
    on="order_id",
    how="left"
)

abt = abt.merge(
    item_features,
    on="order_id",
    how="left"
)

abt = abt.merge(
    order_category,
    on="order_id",
    how="left"
)

In [ ]:
abt.shape

## 31. Task 18: Create Low Review Target Variable

In [ ]:
abt["is_low_review"] = np.where(
    abt["review_score"] <= 2,
    1,
    0
)

In [ ]:
abt["is_low_review"].value_counts()

In [ ]:
abt["is_low_review"].value_counts(normalize=True) * 100

## 32. Task 19: Select Final Columns

In [ ]:
selected_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "order_status",
    "order_year",
    "order_month",
    "order_day",
    "order_day_of_week",
    "order_hour",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days",
    "is_late_delivery",
    "review_score",
    "is_low_review",
    "review_comment_count",
    "has_review_comment",
    "total_payment_value",
    "max_payment_installments",
    "payment_types_count",
    "dominant_payment_type",
    "total_items",
    "total_price",
    "total_freight",
    "unique_products",
    "unique_sellers",
    "main_product_category"
]

abt = abt[selected_columns]

In [ ]:
abt.head()

## 33. Task 20: Inspect Final ABT

In [ ]:
abt.shape

In [ ]:
abt.info()

In [ ]:
abt.isnull().sum().sort_values(ascending=False)

In [ ]:
abt["order_id"].duplicated().sum()

## 34. Task 21: Create ABT Summary Report

In [ ]:
abt_summary = pd.DataFrame({
    "column": abt.columns,
    "data_type": abt.dtypes.astype(str).values,
    "missing_count": abt.isnull().sum().values,
    "missing_percent": (abt.isnull().sum().values / len(abt)) * 100,
    "unique_values": [abt[col].nunique() for col in abt.columns]
})

In [ ]:
abt_summary.to_csv(
    REPORTS_DIR / "lab02_abt_summary.csv",
    index=False
)

## 35. Task 22: Visualize Key Features

In [ ]:
abt["review_score"].value_counts().sort_index().plot(kind="bar")
plt.title("Review Score Distribution")
plt.xlabel("Review Score")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lab02_review_score_distribution.png")
plt.show()

In [ ]:
abt["total_payment_value"].hist(bins=50)
plt.title("Distribution of Total Payment Value")
plt.xlabel("Total Payment Value")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lab02_payment_value_distribution.png")
plt.show()

In [ ]:
abt["customer_state"].value_counts().head(10).plot(kind="bar")
plt.title("Top 10 Customer States by Orders")
plt.xlabel("Customer State")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lab02_top_customer_states.png")
plt.show()

## 36. Task 23: Save Final Processed Dataset

In [ ]:
abt.to_csv(
    PROCESSED_DIR / "olist_orders_abt.csv",
    index=False
)

In [ ]:
(PROCESSED_DIR / "olist_orders_abt.csv").exists()

In [ ]:
abt_check = pd.read_csv(PROCESSED_DIR / "olist_orders_abt.csv")
abt_check.head()